In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-01 12:29:05--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  1.62MB/s    in 0.7s    

2026-07-01 12:29:06 (1.62 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [5]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [7]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [8]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [9]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s]
decode = lambda l : ''.join([itos[i] for i in l])
print(encode("hi there"))
print(decode([46, 47, 1, 58, 46, 43, 56, 43]))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [10]:
data = torch.tensor(encode(text), dtype=torch.long)
data.shape

torch.Size([1115394])

In [11]:
print(data[:1000])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [12]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [13]:
block_size=8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [14]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [15]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
def get_batch(split):
    # generate a small batch of data of inputs x and target y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [14]:
#ix = torch.randint(len(train_data) - 8, (4,))
#ix

In [15]:
#x = torch.stack([train_data[i:i+block_size] for i in ix])
#x

In [16]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [17]:
torch.manual_seed(1337)
class BigramLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets == None:
            loss = None
        else :
            B,T,C = logits.shape #B is batch_size,T is block_size , C is total number of char in the training set
            logits = logits.view(B*T,-1)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits,loss
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [18]:
m = nn.Embedding(65,65)
B,T,C = m(xb).shape
print(B,T,C)

4 8 65


In [19]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.6630, grad_fn=<NllLossBackward0>)

v?vqKGh
txYXcDRSW&jxn;P Kuvbv
Bl:KuMWi;JYlFahlK3u.M,QV;oky3wcxoYPQpbpb
r3Nc 'MQHQ uOfxoxwXIvSE-$&a$K


In [20]:
optimizer = torch.optim.AdamW(m.parameters(), lr=5e-3)

In [21]:
batch_size = 32
for steps in range(100000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss)

tensor(2.5347, grad_fn=<NllLossBackward0>)


In [22]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))



Wheithy,
Thed hovend te
ANCK:
ANCESTharer COMInerd.
NCay
D:

NEd wek;
Wine po!
OFomeashore r.
STESt tthome ne COUSo V:
Ber d shagord,

Henwis usthodecle tond owhacke, acher 's incileirildeshy ome hat; s prn Its STh I mocthees had, nowofrangoland of d.
UTor.

Bot be yovitchireanorisordot, st arte m mas thalded remo ISTUK: merwowofivel y d t wad p! to iswoud hee antrromatoueng nds shacoul
DUFrunn fow'd
Food s s he,
Ay, andd; pe ot mend d e te;
Are pr An cayowhad harlvom styse adausteat onesteromingh future
Yer.
Sisllssoishoree s n fo Bu tis lceashourvias! lllicon, amimyoser t t Ifode m pl himanoulie fsish me m llingasod?
Whe e, e

Th any mouthiturowha orors herith.
Qunilar s, we wamar:
ONCKilan s n:'d checro oond; t pe n toves te byouroutshilir t, hin en se f fr;
BE:
LO:

Anshindit RDot

Pot,

As cen
YONTIteat towe my beeil alu oouil n the lfrt: te we bl,
Whe I l.
Bu abeit d I d w oursermatr ngeaneafls gnd. dennghanound, icourthamane?
Thbat set:
NUS:'sst wiu; ve ighaksirkn whachagarr's

## Attention

In [23]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch time , channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [24]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [36]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [32]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

# efficent ways

##  matrix mul

In [84]:
# for every i th step mathematically sum of ai/i so this is like a lower triagular matrix with 1/row as each element
a = torch.tril(torch.ones(8,8))
a

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [85]:
a = a/torch.sum(a,-1,keepdim=True)
a

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [86]:
a.shape

torch.Size([8, 8])

In [87]:
 xbow2 = a @ x

In [88]:
xbow[1]  ## actually both are same but due to some floating point issue

tensor([[ 1.3488, -0.1396],
        [ 0.8173,  0.4127],
        [-0.1342,  0.4395],
        [ 0.2711,  0.4774],
        [ 0.2421,  0.0694],
        [ 0.0084,  0.0020],
        [ 0.0712, -0.1128],
        [ 0.2527,  0.2149]])

In [91]:
xbow2 == xbow
#xbow2[1]

tensor([[[ True,  True],
         [ True,  True],
         [ True, False],
         [ True,  True],
         [ True, False],
         [False,  True],
         [False, False],
         [ True,  True]],

        [[ True,  True],
         [ True,  True],
         [False,  True],
         [ True,  True],
         [False, False],
         [False, False],
         [False,  True],
         [False,  True]],

        [[ True,  True],
         [ True,  True],
         [False, False],
         [ True,  True],
         [False, False],
         [False,  True],
         [False, False],
         [False, False]],

        [[ True,  True],
         [ True,  True],
         [ True,  True],
         [ True,  True],
         [ True,  True],
         [ True, False],
         [ True, False],
         [ True, False]]])

## softmax

In [79]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

## Self attention

### Visualizing 3D Tensor Multiplication: `(3, 3, 4) @ (3, 4, 3)`

When moving from 2D matrices to 3D tensors, PyTorch performs **Batch Matrix Multiplication**. The key rule to remember is that **the very first dimension is treated as an isolated batch index.** PyTorch runs a parallel assembly line where data in one batch never mixes with another.

---

### 1. The Shape Breakdown

Let's label the dimensions of our tensors:
* **Tensor A:** `(3, 3, 4)` $\rightarrow$ `(Batch Size = 3, Rows = 3, Columns = 4)`
* **Tensor B:** `(3, 4, 3)` $\rightarrow$ `(Batch Size = 3, Rows = 4, Columns = 3)`

Because the batch sizes match (`3` and `3`), PyTorch locks them together. It then performs standard 2D matrix multiplication on the remaining dimensions `(3, 4) @ (4, 3)`. 

The final resulting shape is **`(3, 3, 3)`**.

---

### 2. Numerical Example (Zooming into Batch Index 1)

Instead of looking at all 108 numbers at once, let's look entirely at **Batch Index 1** (the middle layer). PyTorch completely isolates this layer from Layer 0 and Layer 2.

#### **Tensor A at Batch 1 (`A[1]`, Shape 3x4):**
$$
A[1] = \begin{pmatrix} 1 & 0 & 2 & 1 \\ 0 & 3 & 1 & 0 \\ 2 & 1 & 0 & 1 \end{pmatrix}
$$

#### **Tensor B at Batch 1 (`B[1]`, Shape 4x3):**
$$
B[1] = \begin{pmatrix} 2 & 1 & 0 \\ 0 & 2 & 3 \\ 1 & 0 & 1 \\ 3 & 1 & 2 \end{pmatrix}
$$

---

### 3. The 2D Multiplication Assembly Line

We run standard row-meets-column dot products *only* using the isolated matrices above:

* **Row 1 of A[1] visits columns of B[1]:**
  * **Slot (1,1):** $(1\times2) + (0\times0) + (2\times1) + (1\times3) = 2 + 0 + 2 + 3 = \mathbf{7}$
  * **Slot (1,2):** $(1\times1) + (0\times2) + (2\times0) + (1\times1) = 1 + 0 + 0 + 1 = \mathbf{2}$
  * **Slot (1,3):** $(1\times0) + (0\times3) + (2\times1) + (1\times2) = 0 + 0 + 2 + 2 = \mathbf{4}$

* **Row 2 of A[1] visits columns of B[1]:**
  * **Slot (2,1):** $(0\times2) + (3\times0) + (1\times1) + (0\times3) = 0 + 0 + 1 + 0 = \mathbf{1}$
  * **Slot (2,2):** $(0\times1) + (3\times2) + (1\times0) + (0\times1) = 0 + 6 + 0 + 0 = \mathbf{6}$
  * **Slot (2,3):** $(0\times0) + (3\times3) + (1\times1) + (0\times2) = 0 + 9 + 1 + 0 = \mathbf{10}$

* **Row 3 of A[1] visits columns of B[1]:**
  * **Slot (3,1):** $(2\times2) + (1\times0) + (0\times1) + (1\times3) = 4 + 0 + 0 + 3 = \mathbf{7}$
  * **Slot (3,2):** $(2\times1) + (1\times2) + (0\times0) + (1\times1) = 2 + 2 + 0 + 1 = \mathbf{5}$
  * **Slot (3,3):** $(2\times0) + (1\times3) + (0\times1) + (1\times2) = 0 + 3 + 0 + 2 = \mathbf{5}$

---

### 4. The Final Stacked Output

The resulting $3\times3$ matrix from our calculation becomes **Layer 1** of the final 3D tensor. PyTorch repeats this independently for Layer 0 and Layer 2, then stacks them up:

```text
Final Output Tensor (Shape: 3, 3, 3)

             [ Layer 2 (3x3 Matrix from A[2] @ B[2]) ]
             
           [   7     2     4   ] 
           [   1     6    10   ]  <-- Layer 1 (The math we just did)
           [   7     5     5   ]
           
         [ Layer 0 (3x3 Matrix from A[0] @ B[0]) ]

In [94]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False) # what do i contain
query = nn.Linear(C, head_size, bias=False) # what am i looking for 
value = nn.Linear(C, head_size, bias=False) # This is the value of some thing we are loooking for.
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [4]:
class Head(nn.Module):

    def __ini__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_emb,head_size)
        self.query = nn.Linear(n_emb,head_size)
        self.value = nn.Linear(n_emb,head_size)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        # attention scores
        wei = (q @ k.transpose(-2, -1)) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei,dim=-1)
        # weighted aggregation
        v = self.value(x)
        out = wei @ v
        return out
        

In [7]:
# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * (k.shape[-1])**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size) # self attention used to communicate
        self.ffwd = FeedFoward(n_embd) # the feed forward think after communicate
        self.ln1 = nn.LayerNorm(n_embd) #
        self.ln2 = nn.LayerNorm(n_embd) #

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.4109, val loss 4.4016
step 100: train loss 2.6511, val loss 2.6604
step 200: train loss 2.4940, val loss 2.4901
step 300: train loss 2.3905, val loss 2.4022
step 400: train loss 2.3215, val loss 2.3283
step 500: train loss 2.2546, val loss 2.2713
step 600: train loss 2.1963, val loss 2.2059
step 700: train loss 2.1677, val loss 2.1905
step 800: train loss 2.1260, val loss 2.1532
step 900: train loss 2.0774, val loss 2.1114
step 1000: train loss 2.0563, val loss 2.0903
step 1100: train loss 2.0289, val loss 2.0838
step 1200: train loss 1.9993, val loss 2.0486
step 1300: train loss 1.9891, val loss 2.0366
step 1400: train loss 1.9536, val loss 2.0072
step 1500: train loss 1.9369, val loss 2.0013
step 1600: train loss 1.9205, val loss 2.0074
step 1700: train loss 1.9124, val loss 1.9915
step 1800: train loss 1.8805, val loss 1.9749
step 1900: train loss 1.8784, val loss 1.9631
step 2000: train loss 1.8537, val loss 1.9700
step 2100: train loss 1.

### Visualizing a Transformer Block: "Communication followed by Computation"

To understand a Transformer Block without getting lost in the code, let's throw out the math notation and trace **one single word** traveling through a single block. 

Imagine we are processing the sentence: **"The cat sat."** We will follow the word **"cat"** step-by-step.

---

### 1. The Key Variables Explained

Before entering the block, the word **"cat"** is represented as a list of numbers (a vector). 

| Variable Name | What it means in plain English | Example Value |
| :--- | :--- | :--- |
| `n_embd` | The total number of descriptive values used to represent a single word. | `384` |
| `n_head` | How many parallel mini-brains look at the sentence at the same time. | `6` |
| `head_size` | The chunk of memory allocated to each individual mini-brain ($384 \div 6$). | `64` |
| `x` | The running word vector itself, which gets constantly updated. | Tensor of shape `(B, T, 384)` |

---

### 2. The Step-by-Step Timeline of a Word

Incoming Vector "cat"<br>
│<br>
▼<br>
┌───────────────┐
│  LayerNorm 1  │  <--- Clean up and stabilize the numbers
└───────┬───────┘
▼<br>
┌───────────────┐
│ Multi-Head Att│  <--- Communication: 6 mini-brains look for context
└───────┬───────┘
▼<br>
( Highway 1 )   <--- Residual Addition: Original Meaning + Contextual Clues
│
▼<br>
┌───────────────┐
│  LayerNorm 2  │  <--- Clean up the numbers again
└───────┬───────┘
▼<br>
┌───────────────┐
│  FeedForward  │  <--- Computation: Expand 4x to "think" in isolation, then shrink
└───────┬───────┘
▼<br>
( Highway 2 )   <--- Residual Addition: Upgraded Vector + Deeply Thought-Out Logic
│
▼<br>
Final Block Output

#### **Step 1: LayerNorm 1 (`self.ln1`)**
Before "cat" does anything, it passes through Layer Normalization. This adjusts the 384 numbers representing "cat" so they have a stable mean and variance. It prevents values from exploding or shrinking too much.

#### **Step 2: The Attention Machine (`self.sa`) — *The Communication Block***
The cleaned-up numbers for "cat" enter the multi-head attention layer. 
* The 384 numbers are split evenly among **6 parallel heads** (`head_size = 64`). 
* Instead of one giant brain, 6 mini-brains look at the sentence simultaneously to gather different types of clues:
  * *Head 1 asks:* "Is there a verb nearby?" $\rightarrow$ It flags **"sat"**.
  * *Head 2 asks:* "Is there an article nearby?" $\rightarrow$ It flags **"The"**.
* When finished, the 6 heads glue their 64-dimensional results back together (`6 * 64 = 384`) into a unified context vector.

#### **Step 3: The First Highway Connection (`x = x + ...`)**
We perform a **Residual Addition**. We take the original dictionary meaning of "cat" and add it directly to the new contextual clues we just gathered:
$$\text{Dictionary Meaning} + \text{Contextual Clues} = \text{An Upgraded Word Vector}$$
This addition creates a "highway" for gradients during backpropagation, allowing the model to train incredibly fast without losing its learning signal.

#### **Step 4: LayerNorm 2 (`self.ln2`)**
The upgraded "cat" vector is cleaned up one more time to prepare it for individual processing.

#### **Step 5: The FeedForward Machine (`self.ffwd`) — *The Computation Block***
The upgraded, normalized "cat" vector enters the FeedForward station. **Crucially, tokens are completely isolated here; "cat" cannot see "The" or "sat" inside this layer.**
* **The Expansion:** The network projects the 384 numbers up into a larger space of **1,536 numbers** ($4 \times 384$) to give the model "scratchpad space" to calculate intricate logic.
* **The Activation (ReLU):** It filters out negative numbers, forcing the model to make non-linear decisions.
* **The Compression:** It squeezes the data back down to our baseline size of **384 numbers**.

#### **Step 6: The Second Highway Connection (`x = x + ...`)**
Finally, we execute the second residual addition, adding the results of the isolated processing back to our running vector:
$$\text{Upgraded Contextual Vector} + \text{Deeply Thought-Out Logic} = \text{Final Block Output}$$

---

### 3. Summary of the Code Block

When you look at the `forward` function now, it maps directly to this elegant rhythmic dance:

```python
def forward(self, x):
    # 1. Normalize -> Communicate (Attention) -> Add to running highway
    x = x + self.sa(self.ln1(x))
    
    # 2. Normalize -> Compute (FeedForward) -> Add to running highway
    x = x + self.ffwd(self.ln2(x))
    
    return x